# Preproccessing of the dataset

In [ ]:
import pandas as pd
import json
import os
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, ImageOps
import numpy as np
from tqdm import tqdm
from keras.utils import to_categorical


# ---------------------------------------
# Loading the image data
# ---------------------------------------

In [ ]:
train_data_path = '../raw_data/train/train_data/'
train_labels_path = '../raw_data/train/train_labels.json'
test_data_path = '../raw_data/test/test_data/'
test_labels_path = '../raw_data/test/test_labels.json'


In [ ]:
def load_split(labels_path, data_path):
    with open(labels_path) as f:
        data = json.load(f)

    images = pd.DataFrame(data['images']).rename(columns={'id': 'image_id'})
    annotations = pd.DataFrame(data['annotations']).rename(columns={'id': 'egg_id'})
    categories = pd.DataFrame(data['categories']).rename(columns={'id': 'category_id', 'name': 'category_name'})

    df = (images
          .merge(annotations, on='image_id')
          .merge(categories, on='category_id')
          .drop(columns=['license', 'coco_url', 'supercategory']))
    df['path'] = df['file_name'].apply(lambda n: os.path.join(data_path, n))
    return df

train_df = load_split(train_labels_path, train_data_path)
test_df  = load_split(test_labels_path,  test_data_path)

# ---------------------------------------
# Visualisation
# ---------------------------------------

In [ ]:
def show_with_box(row: pd.Series):
    img = Image.open(row['path'])
    x, y, w, h = row['bbox']

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(img)
    ax.add_patch(patches.Rectangle((x, y), w, h, linewidth=2, edgecolor='blue', facecolor='none'))
    ax.axis('off')
    plt.show()

show_with_box(train_df.iloc[10])

In [ ]:
def crop_egg(row: pd.Series, margin: float = 0.2, img_size: int = 224):
    img = ImageOps.exif_transpose(Image.open(row['path'])).convert('RGB')
    x, y, w, h = row['bbox']

    # centre of the egg
    cx, cy = x + w / 2, y + h / 2

    # one side length for both directions: longest egg side + 20% margin on each side
    side = max(w, h) * (1 + 2 * margin)

    left   = cx - side / 2
    top    = cy - side / 2
    right  = cx + side / 2
    bottom = cy + side / 2

    crop = img.crop((left, top, right, bottom))
    return crop.resize((img_size, img_size))

crop_egg(train_df.iloc[10])

### This code is how we get all image_ids that contain 2 or more eggs in one image.

  **`multiple_eggs = train_df.value_counts('image_id').sort_values(ascending=False).head(30)`**

# ---------------------------------------
# One-Hot Encoding, Arrays and Scaling
# ---------------------------------------

In [ ]:
IMG_SIZE    = 224
NUM_CLASSES = 11

def build_arrays(df: pd.DataFrame, img_size: int = IMG_SIZE):
    df = df.reset_index(drop=True)                       # row i in df == X[i]
    X = np.zeros((len(df), img_size, img_size, 3), dtype=np.uint8)

    for i, row in tqdm(df.iterrows(), total=len(df)):
        X[i] = np.asarray(crop_egg(row, img_size=img_size))

    y = to_categorical(x=df['category_id'], num_classes=NUM_CLASSES) # one-hot, shape (n, 11)
    X = X/255.0  # normalize to [0, 1]
    return X, y

In [ ]:
X_train, y_train = build_arrays(train_df)
X_test, y_test = build_arrays(test_df)

In [ ]:
np.savez_compressed(f'../raw_data/crops_{IMG_SIZE}.npz',
                    X_train=X_train,
                    y_train=y_train,
                    X_test=X_test,
                    y_test=y_test
                    )

In [ ]:
train_df.category_name.unique()
